# common_pathogenic_variants.ipynb
Which pathogenic variants are more common in ecDNA+ samples?  
For each sample, get pathogenic variants and aggregate all pathogenic variants into table, run chi2 test and FDR correction on each gene.  
We ran the test on cancer genes mutated at least 4 times.  
We used subset tumor types with at least 1 gene mutant and threw out tumor types without ecDNA.  

RESULT  
Significant difference was obseved in TP53 and CTNNB1.  
There was a possitive correlation between TP53 and ecDNA.  
There was a negative correlation between CTNNB1 and ecDNA.  
TERT showed a significant p-value befre FDR correction but it was no longer significant after the adjustment.  

## import and format
our data into a DataFrame

In [ ]:
import cyvcf2
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
import os
import scipy.stats
from statsmodels.stats.multitest import multipletests

In [ ]:
def parse_CSQ(fields,csq):
    '''
    parse an input string into a nice dictionary.
    For CBTN somatic variant files, CSQ has 73 entries, separated by '|'. 
    Variants may have more than 1 CSQ associated with multiple transcripts, separated by ','.
    '''
    fields = fields.split('|')
    csqs = csq.split(',')
    csqs = [x.split('|') for x in csqs]
    csqs = list(zip(*csqs))
    csqs = [set(x) - {''} for x in csqs]
    #return csqs
    assert len(csqs) == len(fields),f'{len(csqs)},{len(fields)}'
    return dict(zip(fields,csqs))

In [ ]:
def parse_vcf(vcf_path):
    '''
    read a vcf file into a pandas dataframe.
    '''
    all_variants = pd.DataFrame()
    vcf = cyvcf2.VCF(vcf_path)
    csq_header = vcf.get_header_type('CSQ')['Description'].split('Format: ')[-1]
    for variant in vcf:
        values ={
            'REF':variant.REF, 
            'ALT':variant.ALT, 
            'CHROM':variant.CHROM, 
            'start':variant.start, 
            'end':variant.end, 
            'ID':variant.ID,
            'FILTER':variant.FILTER, 
            'QUAL':variant.QUAL,
            'HotSpot':variant.INFO.get('HotSpotAllele'),
            'CSQ':variant.INFO.get('CSQ')
        }
        csq = parse_CSQ(csq_header,values['CSQ'])
        print(len(csq))
        values = csq | values
        #print(values)
        values = pd.DataFrame([values])
        keep = ['CHROM','start','end','REF','ALT','HotSpot','SYMBOL','Consequence','IMPACT','CLIN_SIG']
        values = values[keep]
        all_variants = pd.concat([all_variants,values])
    return all_variants

In [ ]:
def get_vcfs(directory="data/filter/pathogenic"):
    for filename in os.listdir(directory):
        full_path = os.path.join(directory, filename)
        if os.path.isfile(full_path) and filename.endswith(".vcf"):
            yield full_path
def extract_sample_name(filepath):
    sample = os.path.basename(filepath)
    sample = sample.split('.')[0]
    return sample

def get_all_pathogenic_variants(dry_run=False):
    '''
    Read all patient genomes,
    filter for pathogenic variants,
    and aggregate into a single pandas dataframe.
    '''
    big_df = pd.DataFrame()
    i=0
    for vcf in get_vcfs():
        i=i+1
        name = extract_sample_name(vcf)
        print('Now processing sample number',i,':',name)
        variants_df = parse_vcf(vcf)
        variants_df['sample'] = name
        big_df = pd.concat([big_df,variants_df])
        if dry_run == True:
            if i > 5:
                break
    # one row per gene affected
    big_df = big_df.explode('SYMBOL')
    return big_df

In [ ]:
pathogenic_variants_df = get_all_pathogenic_variants(dry_run=False)


In [ ]:
def implode(df):
    return df.groupby(['sample'], as_index=True).agg({
        'SYMBOL': lambda x: set(x)
    })
pathogenic_variants_df = implode(pathogenic_variants_df)

In [ ]:
# how many samples with at least 1 pathogenic variant?
print(len(pathogenic_variants_df))
# how many genes?

In [ ]:
def read_manifest(file='./data/pedpancan/manifest_20250910_143948.csv'):
    df = pd.read_csv(file)
    df=df[df.name.str.endswith('.gz')]
    df['index'] = df.name.map(lambda x: os.path.basename(x).split('.')[0])
    df = df.set_index('index')
    df = df[['Kids First Biospecimen ID']].copy()
    return df

In [ ]:
def read_biosample_table(file='./data/pedpancan/Supplementary Tables.xlsx'):
    return pd.read_excel(file,sheet_name='2. Biosamples',index_col='biosample_id')

In [ ]:
def merge_ecDNA_manifest_variants(ecDNA_df,manifest_df,variants_df):
    manifest_df['file_id'] = manifest_df.index
    manifest_df = manifest_df.set_index('Kids First Biospecimen ID')
    df = ecDNA_df.join(manifest_df,how='inner')
    df['biosample_id'] = df.index
    df = df.set_index('file_id')
    df = df.join(variants_df,how='left')
    df['SYMBOL'] = df['SYMBOL'].apply(lambda x: set() if pd.isna(x) else x)
    # filters
    df = df[df.in_unique_patient_set].copy() # drop biological replicates
    # TODO drop tumor types without ecDNA
    ec_tumor_types = df[df.amplicon_class.map(lambda x: 'ecDNA' in x)].cancer_type.unique()
    df = df[df.cancer_type.isin(ec_tumor_types)]
    return df

In [ ]:
df = merge_ecDNA_manifest_variants(
    read_biosample_table(),
    read_manifest(),
    pathogenic_variants_df
)
df

In [ ]:
len(df.index.unique())
len(df)

## Tests 
detect more common pathogenic variants in ecDNA+ samples by chi2 test, FDR correction.  

In [ ]:
def pathogenic_ccgc_table(file ='./data/databases/Cosmic_CancerGeneCensus_v101_GRCh38.tsv'):
    df = pd.read_table(file)
    df = df.loc[:, ['GENE_SYMBOL']]
    #df = df.set_index('GENE_SYMBOL')
    return df

In [ ]:
ccgc_df = pathogenic_ccgc_table()

In [ ]:
'''
Genes we don't care about:
MIR*
*-AS*
RNU6*
*-DT
'''
path_variant_genes = set(g for gene_set in df['SYMBOL'] for g in gene_set)
cancer_genes = set(ccgc_df['GENE_SYMBOL'].unique())
whitelist = {'H3-3A','H2AC4','H2AC5P','H2BC3','H3C2','H3C3','H4C2','HLA-B','HLA-A','AK4P1','CASC11','DNMT1','MIEN1','PRDM14','RHEB','RIT1','TGFBR1','TSPAN31'}

test_genes = path_variant_genes & (cancer_genes | whitelist)

In [ ]:
def test_cancer_gene(df,gene,verbose=False):
    # ecDNA+ gene+ sample count
    a = ((df['amplicon_class']=='ecDNA') & (df['SYMBOL'].map(lambda s: gene in s))).sum()
    # ecDNA+ gene-
    b = ((df['amplicon_class']=='ecDNA') & (df['SYMBOL'].map(lambda s: not gene in s))).sum()
    # ecDNA- gene+
    c = ((df['amplicon_class']!='ecDNA') & (df['SYMBOL'].map(lambda s: gene in s))).sum()
    # ecDNA- gene-
    d = ((df['amplicon_class']!='ecDNA') & (df['SYMBOL'].map(lambda s: not gene in s))).sum()
    if verbose:
        print(gene, a,b,c,d, a+b+c+d)
    if a + c <4:
        return None
    res = scipy.stats.chi2_contingency([[a,b],[c,d]])
    chi2, p, dof, expected = res
    # code for over/underrepresentation
    #YOUR CODE HERE
    if b*c == 0:
        a += 0.5
        b += 0.5
        c += 0.5
        d += 0.5
    odds_ratio = (a*d) / (b*c)
    enriched = True if odds_ratio >1 else False
    return (chi2, p, enriched)

def test_tt_gene(df,gene):
    ct_df = df[df.SYMBOL.map(lambda x: gene in x)].cancer_type.unique()
    df = df[df.cancer_type.isin(ct_df)]
    return test_cancer_gene(df,gene)

def test_tt_all_cancer_gene(df):
    results = []
    for gene in test_genes:
        res = test_tt_gene(df,gene)
        if res is not None:
            chi2, p, enriched = res
            results.append([gene, chi2, p, enriched])
    result_df = pd.DataFrame(results,columns=['gene', 'chi2', 'pvalue', 'enriched'])
    # multiple hypothesis (FDR) correction
    pvals = result_df['pvalue'].values
    reject,pvals_corrected, _, _ = multipletests(pvals, method='fdr_bh')

    result_df['FDR'] = pvals_corrected
    result_df['significant(FDR<0.05)'] = reject

    return result_df

In [ ]:
improved_df = test_tt_all_cancer_gene(df)
improved_df

# debug

In [ ]:
df[df.SYMBOL.map(lambda x: 'CTNNB1' in x)].cancer_type.unique()

In [ ]:
df[df.SYMBOL.map(lambda x: 'CTNNB1' in x)].groupby('cancer_type').count()['patient_id']

In [ ]:
# CTNNB1 across CPGs samples
df1 = df[df.cancer_type != 'MBL'].copy()
print(pd.crosstab(df1.SYMBOL.map(lambda x: 'CTNNB1' in x),df1.amplicon_class))
print(len(df1))
test_cancer_gene(df1,'CTNNB1')

In [ ]:
# NOTCH1 across HGG only
df2 = df[df.cancer_type == 'HGG'].copy()
print(len(df2))
test_cancer_gene(df2,'TP53')

In [ ]:
df3 = df[df.cancer_type.isin(['HGG','NBL','LGG'])]
print(len(df3))
test_cancer_gene(df3,'NOTCH1')

In [ ]:
df4 = df[df.cancer_type.isin(['MBL', 'GNT', np.nan, 'HGG', 'NBL', 'EMBT', 'LGG', 'EPN'])]
print(len(df4))
test_cancer_gene(df4,'TERT')

In [ ]:
df[df.SYMBOL.map(lambda x: 'TP53' in x)].cancer_type.unique()

In [ ]:
df5 = df[df.cancer_type.isin(['HGG', 'nan', 'LGG', 'MBL', 'SARC', 'EPN', 'PNST', 'EWS', 'MNG','GNT', 'EMBT', 'CPT', 'ATRT', 'GCT'])]
print(len(df5))
test_cancer_gene(df5,'TP53')

In [ ]:
df[df.SYMBOL.map(lambda x: 'H3-3A' in x)].cancer_type.unique()

In [ ]:
df6 = df[df.cancer_type.isin(['nan', 'HGG', 'BENG'])]
print(len(df6))
test_cancer_gene(df6,'H3-3A')